# Multi-Agent Systems with Google Agent Development Kit (ADK)

**Challenge 4**: Hierarchical Orchestration, Lifecycle Callbacks & Loop Workflows**

---

## 🚀 Key Features

* 🤖 **Hierarchical Multi-Agent Architecture:** Root coordinator agent managing specialized sub-agents via delegation and `AgentTool`.
* 🌦️ **Weather Specialist:** Real-time forecast retrieval and coordinate geocoding via the National Weather Service (NWS) API.
* 🔍 **Built-in Google Search Grounding:** Real-time web retrieval using ADK's native `google_search` tool.
* 🎬 **Iterative Film Production Assembly Line:** Autonomous `LoopAgent` pipeline featuring Search, Critique, and Refine agents with state tracking.
* 🛡️ **Lifecycle Callbacks & Moderation:** `before_model` and `after_model` interceptors for location constraints, safety filtering, and request/response telemetry.
* 🧪 **End-to-End Test Suite:** Streamed execution and event logging across multi-turn sessions using `InMemoryRunner` and `AdkApp`.

## Step 1: Install Dependencies

In [2]:
# !pip install google-adk google-genai requests python-dotenv nest-asyncio -q
!pip install "google-adk[extensions]" litellm google-genai requests python-dotenv nest-asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Import Libraries

In [3]:
import os
import json
import requests
import asyncio
import random
import uuid
from typing import Dict, Any, Optional

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Google ADK imports
from google.adk.agents import Agent, LoopAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, ToolContext, google_search
from google.genai.types import Content, Part

from IPython.display import Markdown, display

import vertexai
from vertexai.preview import reasoning_engines

from dotenv import load_dotenv
load_dotenv()

# Set model
MODEL_NAME = "gemini-2.5-flash"


# State management tool matching the slides
def append_to_state(
    tool_context: ToolContext, field: str, response: str
) -> dict:
    """Appends responses into the shared workflow state dictionary."""
    existing_state = tool_context.state.get(field, [])
    tool_context.state[field] = existing_state + [response]
    return {"status": "success"}

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Configuration

Set your API keys here (optional - notebook works without them for common cities)

In [4]:
# API Keys
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Project configuration
PROJECT_ID = "qwiklabs-gcp-02-138827e82db5"
LOCATION = "us-central1"

# Set environment variables for Vertex AI / Google GenAI SDK
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Initialize Vertex AI globally
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("✅ Configuration loaded and Vertex AI initialized")

✅ Configuration loaded and Vertex AI initialized


## Step 2: Define Sub-Agents (Search, Critique, Refine)

In [5]:
# (b) Search Agent: Finds facts, inspiration, and plot premises
search_agent = Agent(
    name="search_agent",
    model=MODEL_NAME,
    description="Searches for real-world background data, historical contexts, and narrative inspirations.",
    instruction="""You are a movie researcher on the film production assembly line.
Use the google_search tool to gather factual details, cinematic references, or plot ideas based on the user's concept.
Summarize your findings clearly for the creative team.""",
    tools=[google_search],
)

# (c) Critique Agent: Evaluates the draft and suggests improvements
critique_agent = Agent(
    name="critique_agent",
    model=MODEL_NAME,
    description="Reviews current film concept drafts and provides constructive critiques and suggestions.",
    instruction="""You are an expert film critic and story editor on the assembly line.
Evaluate the current movie concept draft and search findings.
Identify pacing issues, plot holes, cliché dialogue, or character development gaps.
Provide 2-3 specific, actionable suggestions for improvement.""",
    tools=[append_to_state],
)

# (d) Refine Agent: Rewrites and polishes the script/plot concept
refine_agent = Agent(
    name="refine_agent",
    model=MODEL_NAME,
    description="Rewrites and improves the movie plot concept incorporating critique suggestions.",
    instruction="""You are the lead screenwriter on the assembly line.
Take the existing plot draft and the feedback from the critique_agent.
Rewrite and elevate the film logline, synopsis, and scene beat sheet to address all suggested improvements.
Deliver a polished and compelling final concept.""",
    tools=[append_to_state],
)

## Step 3: Build Loop Agent & Greeter (Root) Agent

In [6]:
# Loop Agent: Iterative writers room refining the script/concept
writers_room = LoopAgent(
    name="writers_room",
    description="Iteratively searches, critiques, and refines the film concept across production cycles.",
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent,
    ],
    max_iterations=2,
)

# Greeter Root Agent: Entry point for user interactions
GREETER_INSTRUCTIONS = """You are the executive producer and greeter for the Film Production Assembly Line.
When a user provides a movie prompt, pitch, or concept:
1. Welcome them to the studio assembly line.
2. Delegate the concept to the writers_room loop team to develop, critique, and polish the story.
3. Deliver the final approved production-ready treatment clearly to the user."""

root_agent = Agent(
    name="greeter",
    model=MODEL_NAME,
    description="Craft a movie plot.",
    instruction=GREETER_INSTRUCTIONS,
    tools=[append_to_state],
    sub_agents=[writers_room],
)

# Wrap root agent into AdkApp & Runner
app = reasoning_engines.AdkApp(agent=root_agent)
runner = InMemoryRunner(agent=root_agent, app_name="Film Production Studio")

print(
    "✅ Film production assembly line agents and LoopAgent created successfully"
)

/var/folders/79/kkzhxd153fs9svz_j_wj0xfr0000gn/T/ipykernel_967/1485925198.py:2: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  writers_room = LoopAgent(
App "Film Production Studio" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


✅ Film production assembly line agents and LoopAgent created successfully


## Step 4: Create Session

In [7]:
user_id = "producer-user-1"
session = app.create_session(user_id=user_id)
session_id = session.get("id") if isinstance(session, dict) else session.id

print(f"🎬 Production Session ID: {session_id}")

/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
App "default-app-name" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after e

🎬 Production Session ID: f5cca14a-914c-4cdd-9e4b-7f1e5b996227


## Step 5: Test Film Production Multi-Agent Assembly Line

In [8]:
def test_film_assembly_line(prompt: str):
    print(f"\n{'='*80}")
    print(f"🎬 NEW FILM PITCH: {prompt}")
    print(f"{'='*80}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=prompt,
        ):
            last_event = event

            # Track agent handoffs and loop steps
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name", "")
                actions = event.get("actions", {})
                if author:
                    print(f"  🔄 [Active Station: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"     ⚙️ Action: {actions}")

        # Render final refined film package
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            print("\n🎞️ FINAL PRODUCTION TREATMENT:")
            display(Markdown(last_event["content"]["parts"][0]["text"]))
        else:
            print("\n⚠️ No final output received from assembly line.")

    except Exception as e:
        print(f"\n❌ Pipeline execution failed: {str(e)}")


# Run Test Pitch
test_film_assembly_line(
    "Pitch a neo-noir sci-fi mystery set in underwater Tokyo in the year 2088."
)


🎬 NEW FILM PITCH: Pitch a neo-noir sci-fi mystery set in underwater Tokyo in the year 2088.


/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


  🔄 [Active Station: greeter]
  🔄 [Active Station: greeter]
     ⚙️ Action: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'writers_room', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Active Station: search_agent]
  🔄 [Active Station: critique_agent]
  🔄 [Active Station: critique_agent]
     ⚙️ Action: {'state_delta': {'critique_agent': ['This concept has a strong foundation, particularly with its vivid setting and compelling neo-noir atmosphere. The blend of sci-fi, mystery, and Japanese folklore is intriguing and offers significant visual and thematic potential. However, to elevate it further, we need to address some areas that could lead to pacing issues or plot inconsistencies.\n\nHere are 3 specific, actionable suggestions for improvement:\n\n1.  **Clarify the Interconnection Between Myth, Technology, and the Cult:** The current draft introduces ancient folklore (Umibōzu/Kappa, shirikodama) and then posits "rogue AI or bio-engineered entit

This revised concept for "Aqua Neon: Tokyo Depths" significantly strengthens the narrative by deepening character motivations, clarifying stakes, and seamlessly intertwining the sci-fi, neo-noir, and mythical elements.